In [1]:
import ast
import os
import glob
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


# =========================================================
# CONFIG
# =========================================================
DATA_FOLDER = r"C:\Users\M S I\OneDrive\Documents\GitHub\Research-Project\data"
TRAIN_FILE = r"C:\Users\M S I\OneDrive\Documents\GitHub\Research-Project\data\synthetic_dataset_with_item.csv"
MODEL_FILE = r"C:\Users\M S I\OneDrive\Documents\GitHub\Research-Project\data\reason_prediction_model.pkl"
CONVERTED_FILE = r"C:\Users\M S I\OneDrive\Documents\GitHub\Research-Project\data\latest_converted_psychopy.csv"
PREDICTION_FILE = r"C:\Users\M S I\OneDrive\Documents\GitHub\Research-Project\data\latest_reason_percentages.csv"


# =========================================================
# HELPERS
# =========================================================
def parse_list_cell(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, (int, float)):
        return [value]

    value = str(value).strip()
    if value == "" or value.lower() == "nan":
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
        return [parsed]
    except Exception:
        return [value]


def get_last_numeric(value):
    arr = parse_list_cell(value)
    nums = []
    for v in arr:
        try:
            nums.append(float(v))
        except Exception:
            pass
    return nums[-1] if len(nums) > 0 else np.nan


def get_last_string(value):
    arr = parse_list_cell(value)
    if len(arr) == 0:
        return None
    return str(arr[-1]).strip()


def standardize_reason_labels(series):
    s = series.astype(str).str.strip().str.lower()

    mapped = s.replace({
        "price": "Price",
        "brand": "Brand",
        "familiarity": "Familiarity"
    })

    return mapped


def normalize_gender(value):
    value = str(value).strip().lower()

    gender_map = {
        "male": "Male",
        "m": "Male",
        "female": "Female",
        "f": "Female"
    }

    return gender_map.get(value, value.title())


def get_user_age():
    while True:
        user_input = input("Enter participant age: ").strip()
        try:
            age = int(user_input)
            if age <= 0:
                print("Please enter a valid positive age.")
                continue
            return age
        except ValueError:
            print("Invalid input. Please enter a whole number for age.")


def get_user_gender():
    while True:
        user_input = input("Enter participant gender (Male/Female): ").strip()
        gender = normalize_gender(user_input)

        if gender in ["Male", "Female"]:
            return gender

        print("Invalid input. Please enter Male or Female.")


# =========================================================
# FIND LATEST CSV
# =========================================================
def get_latest_psychopy_csv(folder_path):
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

    valid_files = []
    for file in csv_files:
        name = os.path.basename(file).lower()

        if "synthetic" in name:
            continue
        if "converted" in name:
            continue
        if "prediction" in name:
            continue
        if "latest_reason" in name:
            continue

        valid_files.append(file)

    if not valid_files:
        raise FileNotFoundError("No valid PsychoPy CSV files found in the folder.")

    latest_file = max(valid_files, key=os.path.getmtime)
    return latest_file


# =========================================================
# TRAIN MODEL
# =========================================================
def train_model(train_file, model_file):
    df = pd.read_csv(train_file)

    required_cols = [
        "brochure_id", "age", "gender", "reason",
        "clicked_item", "reaction_time", "gaze_x", "gaze_y"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Training file is missing columns: {missing}")

    df = df.copy()
    df["reason"] = standardize_reason_labels(df["reason"])
    df["gender"] = df["gender"].astype(str).str.strip().str.title()
    df["clicked_item"] = df["clicked_item"].astype(str).str.strip()

    df = df.dropna(subset=["reason"])
    df = df[df["reason"].isin(["Brand", "Price", "Familiarity"])].reset_index(drop=True)

    feature_cols = [
        "brochure_id", "age", "gender", "clicked_item",
        "reaction_time", "gaze_x", "gaze_y"
    ]
    X = df[feature_cols].copy()
    y = df["reason"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    numeric_features = ["brochure_id", "age", "reaction_time", "gaze_x", "gaze_y"]
    categorical_features = ["gender", "clicked_item"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            random_state=42,
            class_weight="balanced"
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("======================================")
    print("MODEL TRAINING RESULTS")
    print("======================================")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

    joblib.dump(model, model_file)
    print(f"Model saved to: {model_file}\n")

    return model


# =========================================================
# CONVERT RAW PSYCHOPY FILE TO TIDY FORMAT
# ACCORDING TO YOUR DATASET STRUCTURE
# =========================================================
def convert_psychopy_to_tidy(psychopy_file, age_value, gender_value, output_file):
    raw_df = pd.read_csv(psychopy_file)

    if raw_df.empty:
        raise ValueError("The PsychoPy CSV file is empty.")

    participant_value = "Unknown"
    if "participant" in raw_df.columns:
        val = raw_df["participant"].dropna()
        if not val.empty:
            participant_value = str(val.iloc[0]).strip()

    # Mapping based on your actual dataset
    brochure_blocks = [
        {
            "brochure_id": 1,
            "time_col": "mouse_5.time",
            "x_col": "mouse_5.x",
            "y_col": "mouse_5.y",
            "clicked_col": "mouse_5.clicked_name"
        },
        {
            "brochure_id": 2,
            "time_col": "mouse_4.time",
            "x_col": "mouse_4.x",
            "y_col": "mouse_4.y",
            "clicked_col": "mouse_4.clicked_name"
        },
        {
            "brochure_id": 3,
            "time_col": "mouse_3.time",
            "x_col": "mouse_3.x",
            "y_col": "mouse_3.y",
            "clicked_col": "mouse_3.clicked_name"
        },
        {
            "brochure_id": 4,
            "time_col": "mouse.time",
            "x_col": "mouse.x",
            "y_col": "mouse.y",
            "clicked_col": "mouse.clicked_name"
        },
        {
            "brochure_id": 5,
            "time_col": "mouse_2.time",
            "x_col": "mouse_2.x",
            "y_col": "mouse_2.y",
            "clicked_col": "mouse_2.clicked_name"
        }
    ]

    tidy_rows = []

    for block in brochure_blocks:
        brochure_id = block["brochure_id"]
        x_col = block["x_col"]
        y_col = block["y_col"]
        t_col = block["time_col"]
        c_col = block["clicked_col"]

        if c_col not in raw_df.columns:
            print(f"Skipping brochure {brochure_id}: missing column {c_col}")
            continue

        valid_rows = raw_df[raw_df[c_col].notna()]
        if valid_rows.empty:
            print(f"Skipping brochure {brochure_id}: no clicked item found")
            continue

        selected_row = valid_rows.iloc[0]

        clicked_item = get_last_string(selected_row[c_col])
        reaction_time = get_last_numeric(selected_row[t_col]) if t_col in raw_df.columns else np.nan
        gaze_x = get_last_numeric(selected_row[x_col]) if x_col in raw_df.columns else np.nan
        gaze_y = get_last_numeric(selected_row[y_col]) if y_col in raw_df.columns else np.nan

        tidy_rows.append({
            "participant": participant_value,
            "brochure_id": brochure_id,
            "age": age_value,
            "gender": gender_value,
            "reaction_time": reaction_time,
            "gaze_x": gaze_x,
            "gaze_y": gaze_y,
            "clicked_item": clicked_item
        })

    tidy_df = pd.DataFrame(tidy_rows)

    if tidy_df.empty:
        raise ValueError("No valid brochure data was extracted from the latest PsychoPy CSV.")

    # keep order
    tidy_df = tidy_df.sort_values("brochure_id").reset_index(drop=True)

    tidy_df.to_csv(output_file, index=False)

    print("======================================")
    print("CONVERTED LATEST PSYCHOPY FILE")
    print("======================================")
    print(tidy_df)
    print(f"\nConverted file saved to: {output_file}\n")

    return tidy_df


# =========================================================
# PREDICT PERCENTAGES
# =========================================================
def predict_reason_percentages(tidy_df, model_file, output_file):
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Model file not found: {model_file}")

    model = joblib.load(model_file)

    X_new = tidy_df[
        ["brochure_id", "age", "gender", "clicked_item", "reaction_time", "gaze_x", "gaze_y"]
    ].copy()

    probs = model.predict_proba(X_new)
    preds = model.predict(X_new)
    class_names = model.named_steps["classifier"].classes_

    prob_df = pd.DataFrame(
        probs * 100,
        columns=[f"{cls}_Percentage" for cls in class_names]
    )

    final_df = tidy_df.copy()
    final_df["Predicted_Reason"] = preds

    for col in prob_df.columns:
        final_df[col] = prob_df[col].round(2)

    final_df.to_csv(output_file, index=False)

    print("======================================")
    print("PREDICTION OUTPUT")
    print("======================================")
    print(final_df)

    print("\n======================================")
    print("REASON INFLUENCE REPORT")
    print("======================================\n")

    for _, row in final_df.iterrows():
        print(f"Participant      : {row['participant']}")
        print(f"Brochure ID      : {row['brochure_id']}")
        print(f"Clicked Item     : {row['clicked_item']}")
        print(f"Age              : {row['age']}")
        print(f"Gender           : {row['gender']}")
        print(f"Reaction Time    : {row['reaction_time']:.6f}" if pd.notna(row["reaction_time"]) else "Reaction Time    : NaN")
        print(f"Gaze X           : {row['gaze_x']:.6f}" if pd.notna(row["gaze_x"]) else "Gaze X           : NaN")
        print(f"Gaze Y           : {row['gaze_y']:.6f}" if pd.notna(row["gaze_y"]) else "Gaze Y           : NaN")

        print("\nThese are the percentages of reasons that affected the final decision:")

        if "Brand_Percentage" in final_df.columns:
            print(f"   Brand Influence       : {row['Brand_Percentage']:.2f}%")
        if "Familiarity_Percentage" in final_df.columns:
            print(f"   Familiarity Influence : {row['Familiarity_Percentage']:.2f}%")
        if "Price_Percentage" in final_df.columns:
            print(f"   Price Influence       : {row['Price_Percentage']:.2f}%")

        print(f"\nFinal Decision Most Influenced By : {row['Predicted_Reason']}")
        print("\n--------------------------------------\n")

    print(f"Prediction file saved to: {output_file}\n")

    return final_df


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    # Ask user for participant details
    age_value = get_user_age()
    gender_value = get_user_gender()

    # Train model on synthetic dataset
    train_model(TRAIN_FILE, MODEL_FILE)

    # Find latest PsychoPy CSV automatically
    latest_csv = get_latest_psychopy_csv(DATA_FOLDER)
    print("Latest PsychoPy CSV found:")
    print(latest_csv)
    print()

    # Convert latest PsychoPy file using user-entered age and gender
    tidy_df = convert_psychopy_to_tidy(
        psychopy_file=latest_csv,
        age_value=age_value,
        gender_value=gender_value,
        output_file=CONVERTED_FILE
    )

    # Predict reason percentages
    results_df = predict_reason_percentages(
        tidy_df=tidy_df,
        model_file=MODEL_FILE,
        output_file=PREDICTION_FILE
    )

Enter participant age:  23
Enter participant gender (Male/Female):  female


MODEL TRAINING RESULTS
Accuracy: 0.595

Classification Report:

              precision    recall  f1-score   support

       Brand       0.72      0.67      0.70       106
 Familiarity       0.49      0.49      0.49        45
       Price       0.46      0.53      0.49        49

    accuracy                           0.59       200
   macro avg       0.56      0.56      0.56       200
weighted avg       0.61      0.59      0.60       200

Model saved to: C:\Users\M S I\OneDrive\Documents\GitHub\Research-Project\data\reason_prediction_model.pkl

Latest PsychoPy CSV found:
C:\Users\M S I\OneDrive\Documents\GitHub\Research-Project\data\740635_Cheese_2026-03-09_18h50.17.997.csv

Skipping brochure 4: missing column mouse.clicked_name
CONVERTED LATEST PSYCHOPY FILE
  participant  brochure_id  age  gender  reaction_time    gaze_x    gaze_y  \
0      740635            1   23  Female       4.802969 -0.095370 -0.021296   
1      740635            2   23  Female       2.557286  0.470370 -0.0027